# Data preparation (human)

# 1. Import librairies

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import re
from collections import Counter
import matplotlib.pyplot as plt
import os

c:\Users\lvber\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Import AI detector

In [30]:
model_path = "Oxidane/tmr-ai-text-detector"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 928.59it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


# 2. Import victorian texts

This first function, `read_and_clean_txt` is used to read a `.txt`, to remove the very beginning and the very end, which corresponds to the details that the work comes from the Gutemberg website (marked by ***) to keep only the text.

In [ ]:
def read_and_clean_txt(path):

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    if text.count("***") != 4: # just to chexk that there is no *** in the text
        return False

    positions = []
    start = 0

    while True:
        i = text.find("***", start)
        if i == -1:
            break
        positions.append(i)
        start = i + 3

    debut = positions[1] + 3  
    fin = positions[2]      
    text = text[debut:fin]

    text = re.sub(r'\n(?!\n)', '', text) # the file is full of line breaks. These don't correspond to anything unless there are two of them
    print('There is', text.count("\n"), 'paragraphs in', path)
    return(text)

This function `cut_text_in_segments`divides the text into readable chunks of at least 1000 characters, preferably at the end of a line.

In [32]:
def cut_text_in_segments(text):

    segments = []
    start = 0
    min_size = 1000

    while start < len(text):
        # Find where to end this segment
        end = start + min_size
        if end >= len(text):
            # if we go past the end of the text, take everything
            segments.append(text[start:])
            break
        
        # look for the next '\n' after at least min_size characters
        next_newline = text.find('\n', end)
        if next_newline == -1:
            # no '\n' left, take until the end
            segments.append(text[start:])
            break
        
        # cut up to the next '\n'
        segments.append(text[start:next_newline+1])
        start = next_newline + 1 

    return segments

Finally, this script reads text files, cuts them into segments of approximately 1000 characters, associates each segment with its author, and then stores them in a CSV.

In [ ]:
dossier = 'texts'
data = []

fichiers = [f for f in os.listdir(dossier) if os.path.isfile(os.path.join(dossier, f))]

for fichier in fichiers:
    chemin = os.path.join(dossier, fichier)
    text = read_and_clean_txt(chemin)
    segments = cut_text_in_segments(text)

    segments = [seg.replace('\n', ' ') for seg in segments]
    segments = segments[2:-2]
    segments_trie = sorted(segments, key=lambda x: abs(len(x) - 1000))
    segments_final = segments_trie[:50]

    print('The longest paragraph has', len(segments_final[49]), 'signs.', '\n')

    titre = fichier.split(' - ')[0]
    auteur = fichier.split(' - ')[1].split('.')[0]
    
    for seg in segments_final:
        data.append({'texte': seg, 'titre':titre, 'auteur': auteur})

df = pd.DataFrame(data)
df.to_csv('victorian_texts_human.csv', index=False, encoding='utf-8')
df

There is 890 paragraphs in texts\Alice's Adventures in Wonderland - Lewis Carroll.txt
The longest paragraph has 1067 signs. 

There is 3751 paragraphs in texts\Far from the Madding Crowd - Thomas Hardy.txt
The longest paragraph has 1015 signs. 

There is 4173 paragraphs in texts\Great Expectations - Charles Dickens.txt
The longest paragraph has 1016 signs. 

There is 5185 paragraphs in texts\Middlemarch - George Eliot.txt
The longest paragraph has 1020 signs. 

There is 3447 paragraphs in texts\North and South - Elizabeth Cleghorn Gaskell.txt
The longest paragraph has 1030 signs. 

There is 2620 paragraphs in texts\The Adventures of Sherlock Holmes - Arthur Conan Doyle.txt
The longest paragraph has 1028 signs. 

There is 1624 paragraphs in texts\The Picture of Dorian Gray - Oscar Wilde.txt
The longest paragraph has 1046 signs. 

There is 404 paragraphs in texts\The Time Machine - H G Wells.txt
The longest paragraph has 1366 signs. 

There is 1600 paragraphs in texts\Treasure Island - R

,texte,titre,auteur
0,“The twinkling of the _what?_” said the King. ...,Alice's Adventures in Wonderland,Lewis Carroll
1,The first thing she heard was a general chorus...,Alice's Adventures in Wonderland,Lewis Carroll
2,"“Nobody asked _your_ opinion,” said Alice. “Wh...",Alice's Adventures in Wonderland,Lewis Carroll
3,"The executioner’s argument was, that you could...",Alice's Adventures in Wonderland,Lewis Carroll
4,“I haven’t the least idea what you’re talking ...,Alice's Adventures in Wonderland,Lewis Carroll
...,...,...,...
495,"“Oh!” she replied, “I don’t wish to limit his ...",Wuthering Heights,Emily Brontë
496,“It’s a nice place for a young man to fix his ...,Wuthering Heights,Emily Brontë
497,"“Your worthless friend!” I answered, warmly: “...",Wuthering Heights,Emily Brontë
498,My landlord halloed for me to stop ere I reach...,Wuthering Heights,Emily Brontë


# 3. Test

This part will be done elsewhere, it was just to check if the Higging face model were right.

In [40]:
ai = 0

for index, rows in df.iterrows():
    inputs = tokenizer(rows['texte'], return_tensors="pt", truncation=True, max_length=512, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)

    ai_probability = probs[0][1].item()

    is_ai = ai_probability > 0.5

    if is_ai:
        ai += 1
        #display(rows[0])

print(f'Total : {(ai/len(df)*100):.1f}', '% AI')

Total : 1.8 % AI


Total : 1.8 % AI